# 🐔 AgriCare AI — Poultry Diagnostic Engine

This notebook runs the full poultry health diagnostic system:
1. Installs required libraries
2. Loads the verified disease knowledge base (KALRO/IITA)
3. Builds the AI diagnostic engine with multilingual support
4. Launches an interactive session where you type symptoms and get diagnoses

**Supported Languages:** English, Hausa, Yoruba, Igbo, Nigerian Pidgin

---
### How to run
- **Google Colab:** `Runtime` → `Run all`
- **VSCode:** Select a Python kernel, then `Run All`
- **Jupyter:** `Cell` → `Run All`

> The first run downloads the AI model (~120 MB). This is automatic and only happens once.

---
## Step 1: Install Libraries

In [1]:
import subprocess, sys
print('📦 Installing required libraries...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'faiss-cpu', 'numpy', 'torch', 'transformers'])
print('✅ All libraries installed.')

📦 Installing required libraries...
✅ All libraries installed.


---
## Step 2: Load the Knowledge Base
Loads the verified poultry disease database — 15 diseases with symptoms, advice, and severity levels in 5 languages.

In [2]:
import json, os

# Try loading from file, fall back to inline data
KB_PATH = 'data/knowledge_base.json'
if os.path.exists(KB_PATH):
    with open(KB_PATH, 'r', encoding='utf-8') as f:
        KNOWLEDGE_BASE = json.load(f)
    print(f'✅ Loaded {len(KNOWLEDGE_BASE)} diseases from {KB_PATH}')
else:
    # Inline fallback for Colab (works without the JSON file)
    KNOWLEDGE_BASE = [
        {"id":"newcastle","names":{"en":"Newcastle Disease","ha":"Cutar Newcastle","yo":"Àrùn Newcastle","ig":"Ọrịa Newcastle","pcm":"Newcastle Disease"},"severity":"CRITICAL","symptoms":{"en":["gasping","twisted neck","green diarrhea","sudden death","loss of appetite","drooping wings","coughing","nervous signs"],"ha":["haki","karkacewar wuya","zawo mai kore","mutuwar kwatsam","rashin cin abinci"],"yo":["mímí líle","ọrùn yíyípo","ìgbẹ́ àwọ̀ ewé","ikú òjijì"],"ig":["iku ume","olu gbagọrọ agbagọ","afọ ọsịsa akwụkwọ ndụ","ọnwụ mberede"],"pcm":["breathing hard","neck twist","green shit","sudden death"]},"advice":{"en":"VACCINATE at Day 1 (eye drop) and Week 3-4 (LaSota in water). Isolate sick birds IMMEDIATELY. Call a veterinarian TODAY.","ha":"YI RIGAKAFI a Rana 1 (a ido) da Mako 3-4 (LaSota a ruwa). Ware tsuntsaye marasa lafiya YANZU.","yo":"Ṣe AJESARA ní Ọjọ́ kìíní (ojú) àti Ọ̀sẹ̀ kẹta si kẹrin (LaSota nínú omi).","ig":"GBAA ỌGWỤ MGBOCHI n'Ụbọchị 1 (anya) na Izuụka 3-4 (LaSota na mmiri).","pcm":"VACCINATE for Day 1 (eye drop) and Week 3-4 (LaSota inside water). Separate sick birds NOW NOW."},"escalation_words":["gasping","twisted neck","sudden death","mutuwar kwatsam","haki","coughing"]},
        {"id":"avian_influenza","names":{"en":"Avian Influenza (Bird Flu)","ha":"Murar Tsuntsaye (Bird Flu)","yo":"Àrùn Àfòìmọ̀ (Bird Flu)","ig":"Ọrịa Nnụnụ (Bird Flu)","pcm":"Bird Flu"},"severity":"CRITICAL","symptoms":{"en":["sudden death without signs","swollen head","blue comb/wattles","blood-tinged discharge","drop in egg production"],"ha":["mutuwar kwatsam ba tare da alamomi ba","kumburin kai","tsefe mai shuɗi"],"yo":["ikú òjijì láìsí àmì","wíwú orí","àwọ̀ búlúù"],"ig":["ọnwụ mberede na-enweghị ihe ịrịba ama","isi fụrụ akpụ"],"pcm":["sudden death without signs","head swell","blue comb"]},"advice":{"en":"REPORT TO AUTHORITIES. Do not touch sick or dead birds. Call a government vet immediately.","ha":"KAIS DA HUKUMA. Kada ka taba tsuntsaye marasa lafiya ko matattu.","yo":"JÁBỌ̀ FÚN ÀWỌN ALÁṢẸ. Má ṣe fọwọ́ kan ẹyẹ aláìsàn.","ig":"KPỌTỤRỤ NDỊ ỌCHỊCHỊ. Emetụla nnụnụ na-arịa ọrịa aka.","pcm":"REPORT TO GOVERNMENT. No touch sick or dead birds."},"escalation_words":["sudden death","blood","blue comb","swollen head","bird flu"]},
        {"id":"coccidiosis","names":{"en":"Coccidiosis","ha":"Coccidiosis","yo":"Àrùn Coccidiosis","ig":"Ọrịa Coccidiosis","pcm":"Coccidiosis"},"severity":"HIGH","symptoms":{"en":["bloody droppings","weakness","pale comb","reduced feed intake","huddling","weight loss","ruffled feathers"],"ha":["zawo mai jini","rauni","kodan tsefe","rage cin abinci"],"yo":["ìgbẹ́ ẹlẹjẹ","àìlera","àwọ̀ funfun"],"ig":["nsị ọbara","adịghị ike","mbo icha"],"pcm":["bloody shit","weak body","pale comb","no dey chop well"]},"advice":{"en":"Buy AMPROLIUM from vet shop. Give 1g/2L water for 5-7 days. Keep litter dry.","ha":"Sayi AMPROLIUM daga shagon likitan dabbobi. Ba gram 1/lita 2 na ruwa tsawon kwanaki 5-7.","yo":"Ra AMPROLIUM láti ilé ìtajà oògùn. Fún gram 1/lítà omi 2 fún ọjọ́ 5-7.","ig":"Zụta AMPROLIUM n'ụlọ ahịa ọgwụ. Nye gram 1/lita mmiri 2 maka ụbọchị 5-7.","pcm":"Buy AMPROLIUM from vet shop. Give 1g/2L water for 5-7 days."},"escalation_words":["bloody droppings","zawo mai jini","bloody shit","jini","ọbara"]},
        {"id":"gumboro","names":{"en":"Gumboro Disease (IBD)","ha":"Cutar Gumboro","yo":"Àrùn Gumboro","ig":"Ọrịa Gumboro","pcm":"Gumboro"},"severity":"HIGH","symptoms":{"en":["white watery diarrhea","depression","ruffled feathers","dehydration","vent pecking","sudden drop in appetite"],"ha":["zawo mai ruwa farin","baƙin ciki","rashin ruwa"],"yo":["ìgbẹ́ omi funfun","ìbànújẹ́","àìní omi ara"],"ig":["nsị ọcha mmiri mmiri","ịda mbà","akpọnwụ mmiri"],"pcm":["white watery shit","depression","body dry","appetite drop suddenly"]},"advice":{"en":"VACCINATE chicks at Day 14-21. Isolate sick birds. Support with electrolytes in water.","ha":"YI RIGAKAFIN tsuntsaye a rana 14-21. Ware marasa lafiya. Ba su ruwa mai wutan lantarki.","yo":"Ṣe ÀJẸSÁRA fún àwọn ọmọ adìẹ ní ọjọ́ 14-21.","ig":"GBAA ỌGWỤ MGBOCHI ụmụ ọkụkọ n'ụbọchị 14-21.","pcm":"VACCINATE chicks for Day 14-21. Separate sick ones. Give dem water with electrolytes."},"escalation_words":["white watery","dehydration","body dry"]},
        {"id":"fowl_pox","names":{"en":"Fowl Pox","ha":"Cutar Fowl Pox","yo":"Àrùn Fowl Pox","ig":"Ọrịa Fowl Pox","pcm":"Fowl Pox"},"severity":"MEDIUM","symptoms":{"en":["wart-like bumps on comb/wattles","sores in mouth","reduced egg production","loss of appetite","scabs on skin"],"ha":["kumburi kamar wart akan tsefe","raunuka a baki"],"yo":["àpá olóòórùn lórí àwò ẹyẹ","ọgbẹ́ nínú ẹnu"],"ig":["mkomi yiri wart n'elu mbo","ọnya n'ọnụ"],"pcm":["bumps like wart for comb","sore for mouth"]},"advice":{"en":"VACCINATE with fowl pox vaccine. No specific treatment. Provide soft food and clean water.","ha":"YI RIGAKAFI da maganin fowl pox. Babu takamaiman magani.","yo":"Ṣe ÀJẸSÁRA pẹ̀lú oògùn fowl pox. Kò sí oògùn pàtó.","ig":"GBAA ỌGWỤ MGBOCHI fowl pox. Enweghị ọgwụgwọ pụrụ iche.","pcm":"VACCINATE with fowl pox vaccine. No special medicine."},"escalation_words":[]}
    ]
    print(f'✅ Loaded {len(KNOWLEDGE_BASE)} diseases from inline data')
    print('   (For all 15 diseases, place data/knowledge_base.json next to this notebook)')

✅ Loaded 5 diseases from inline data
   (For all 15 diseases, place data/knowledge_base.json next to this notebook)


---
## Step 3: Build the Diagnostic Engine
Creates the core engine using a multilingual AI model to match symptoms to diseases across all 5 supported languages.

In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import Dict, List, Optional, Tuple, Any

class AgriCareEngine:
    """Core AI diagnostic engine for poultry health."""

    def __init__(self, knowledge: list, model_name='paraphrase-multilingual-MiniLM-L12-v2'):
        self.knowledge = knowledge
        print(f'🧠 Loading AI model: {model_name} ...')
        self.brain = SentenceTransformer(model_name)
        self.doc_texts = []
        self.doc_diseases = []
        for disease in self.knowledge:
            for lang, symptoms in disease.get('symptoms', {}).items():
                for symptom in symptoms:
                    self.doc_texts.append(f'Lang:{lang} Sx:{symptom}')
                    self.doc_diseases.append(disease)
        vectors = self.brain.encode(self.doc_texts)
        dim = vectors.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(vectors).astype('float32'))
        print(f'✅ Engine ready! Indexed {len(self.doc_texts)} symptom entries across {len(self.knowledge)} diseases.')

    def detect_language(self, text: str) -> str:
        LANG_WORDS = {
            'ha': ['kaji','kajina','kaza','kazzar','cutar','alamun','magani','ruwa','abinci','gaggawa','zawo','jini','kore','fari','baki'],
            'yo': ['adìẹ','arun','àmì','oògùn','omi','oúnjẹ','pàjáwìrì','ìgbẹ́','ẹ̀jẹ̀','ewé','funfun','dudu'],
            'ig': ['ọkụkọ','ọrịa','ihe','ọgwụ','mmiri','nri','nsị','ọbara','akwụkwọ','ọcha','oji'],
            'pcm': ['dey','wey','go','fit','chop','sabi','wetin','abeg','shit','blood','belle','don']
        }
        t = text.lower()
        scores = {lang: sum(1 for w in words if w in t) for lang, words in LANG_WORDS.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] >= 1 else 'en'

    def detect_intent(self, text: str, lang: str = 'en') -> str:
        INTENTS = {
            'emergency': ['emergency','help me','urgent','dying','mutuwa','pàjáwìrì','ngwa ngwa'],
            'treatment': ['treat','medicine','cure','drugs','magani','oògùn','ọgwụ']
        }
        text_lower = text.lower()
        for intent, patterns in INTENTS.items():
            if any(pat in text_lower for pat in patterns):
                return intent
        return 'diagnose'

    def find_disease(self, text: str, lang: str = 'en') -> Optional[Dict]:
        text_lower = text.lower()
        words = set(text_lower.split())
        for disease in self.knowledge:
            for s in disease['symptoms'].get(lang, []):
                s_lower = s.lower()
                if s_lower in text_lower:
                    return disease
                s_words = set(s_lower.split())
                if len(words.intersection(s_words)) >= 2:
                    return disease
            for ew in disease.get('escalation_words', []):
                if ew.lower() in text_lower:
                    return disease
        try:
            query_vec = self.brain.encode([f'Lang:{lang} Query:{text}'])
            _, indices = self.index.search(np.array(query_vec).astype('float32'), k=1)
            return self.doc_diseases[indices[0][0]]
        except:
            return None

    def triage(self, text: str, disease: Optional[Dict], intent: str) -> Tuple[str, bool]:
        if intent == 'emergency' or (disease and disease.get('severity') == 'CRITICAL'):
            return 'RED', True
        if disease:
            sev = disease.get('severity')
            if sev == 'HIGH': return 'ORANGE', False
            if sev == 'MEDIUM': return 'YELLOW', False
        return 'GREEN', False

    def build_response(self, disease: Optional[Dict], urgency: str, lang: str, intent: str) -> str:
        if not disease:
            return 'I cannot identify the condition. Please consult a veterinarian immediately.'
        name = disease['names'].get(lang, disease['names']['en'])
        advice = disease['advice'].get(lang, disease['advice']['en'])
        templates = {'RED': '🚨 EMERGENCY: ', 'ORANGE': '⚠️ URGENT: ', 'YELLOW': '📋 INFO: ', 'GREEN': '✅ ADVICE: '}
        prefix = templates.get(urgency, '📋 ')
        return f'{prefix}{name}\n\nAdvice: {advice}'

    def process(self, text: str) -> Dict[str, Any]:
        lang = self.detect_language(text)
        intent = self.detect_intent(text, lang)
        disease = self.find_disease(text, lang)
        urgency, escalate = self.triage(text, disease, intent)
        answer = self.build_response(disease, urgency, lang, intent)
        return {
            'language': lang,
            'intent': intent,
            'disease': disease['id'] if disease else None,
            'urgency': urgency,
            'escalate': escalate,
            'answer': answer
        }

engine = AgriCareEngine(KNOWLEDGE_BASE)

🧠 Loading AI model: paraphrase-multilingual-MiniLM-L12-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Engine ready! Indexed 94 symptom entries across 5 diseases.


---
## Step 4: Interactive Diagnostic Session
Type any symptom description and the engine will diagnose it in real time. Type `quit` to end.

In [ ]:
from datetime import datetime

print('=' * 60)
print('🐔 POULTRY HEALTH DIAGNOSTIC SYSTEM')
print('   Powered by Verified Veterinary Data (KALRO/IITA)')
print('=' * 60)
print(f'📅 Session Started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()
print('📝 Describe your poultry concern below.')
print('🌍 Supported: English | Hausa | Yoruba | Igbo | Pidgin')
print('⚠️  Type \'quit\' to end session.')
print()

query_count = 0

while True:
    farmer_query = input('👨‍🌾 Farmer Query: ')

    if farmer_query.strip().lower() in ['quit', 'exit', 'stop']:
        print(f'\n{"=" * 60}')
        print(f'📊 Session Summary: {query_count} queries processed')
        print(f'👋 Session ended. Thank you for using Poultry Health AI.')
        break

    if not farmer_query.strip():
        continue

    query_count += 1
    result = engine.process(farmer_query)

    print(f'\n{"─" * 60}')
    print(f'📋 DIAGNOSTIC REPORT #{query_count}')
    print(f'{"─" * 60}')
    print(f'🌍 Language:     {result["language"]}')
    print(f'🎯 Query Intent: {result["intent"]}')
    print(f'🦠 Condition:    {result["disease"] or "No specific condition identified"}')
    print(f'🚦 Severity:     {result["urgency"]}')
    print(f'⚠️  HITL Alert:  {"🟢 ESCALATED - Vet notification triggered" if result["escalate"] else "⚪ Not escalated"}')
    print(f'{"─" * 60}')
    print(f'🤖 Recommendation:\n{result["answer"]}')
    print(f'{"─" * 60}\n')

🐔 POULTRY HEALTH DIAGNOSTIC SYSTEM
   Powered by Verified Veterinary Data (KALRO/IITA)
📅 Session Started: 2026-05-13 17:36:33

📝 Describe your poultry concern below.
🌍 Supported: English | Hausa | Yoruba | Igbo | Pidgin
⚠️  Type 'quit' to end session.

